# Baseline XGBoost–SARIMA
A leak-safe hybrid baseline. XGBoost learns one global panel model; SARIMA models each sufficiently long Store–Dept series. The validation blend is selected using Kaggle WMAE.

In [ ]:
%pip install -q "xgboost>=3,<4" "statsmodels>=0.14,<1" "joblib>=1.4,<2"

In [ ]:
from pathlib import Path
import warnings, joblib, numpy as np, pandas as pd
from xgboost import XGBRegressor
from statsmodels.tsa.statespace.sarimax import SARIMAX
warnings.filterwarnings('ignore')
DATA_DIR = Path('/content/drive/MyDrive/walmart_competition_data') if Path('/content').exists() else Path('../../../data')
OUTPUT_DIR = Path('/content/drive/MyDrive/walmart_models') if Path('/content').exists() else Path('artifacts')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_WEEKS, MIN_SARIMA_POINTS = 13, 80
SARIMA_ORDER, SEASONAL_ORDER = (1,0,1), (0,1,1,52)

In [ ]:
train = pd.read_csv(DATA_DIR/'train.csv', parse_dates=['Date'])
stores = pd.read_csv(DATA_DIR/'stores.csv')
train = train.merge(stores, on='Store', how='left')
def features(df):
    x=df.copy(); d=x.Date
    x['year']=d.dt.year; x['week']=d.dt.isocalendar().week.astype(int)
    x['month']=d.dt.month; x['week_sin']=np.sin(2*np.pi*x.week/52)
    x['week_cos']=np.cos(2*np.pi*x.week/52); x['Type']=x.Type.map({'A':0,'B':1,'C':2})
    return x[['Store','Dept','IsHoliday','Type','Size','year','week','month','week_sin','week_cos']].astype(float)
def wmae(y,p,h):
    w=np.where(np.asarray(h),5.,1.); return np.average(np.abs(np.asarray(y)-p),weights=w)
dates=np.sort(train.Date.unique()); cut=dates[-VALIDATION_WEEKS]
tr=train[train.Date<cut].copy(); va=train[train.Date>=cut].copy()

In [ ]:
xgb=XGBRegressor(n_estimators=900,max_depth=8,learning_rate=.04,subsample=.85,colsample_bytree=.85,objective='reg:absoluteerror',tree_method='hist',random_state=42)
xgb.fit(features(tr),tr.Weekly_Sales,sample_weight=np.where(tr.IsHoliday,5.,1.))
px=xgb.predict(features(va)); ps=px.copy()
for key, idx in va.groupby(['Store','Dept']).groups.items():
    y=tr[(tr.Store==key[0])&(tr.Dept==key[1])].sort_values('Date').Weekly_Sales
    if len(y)<MIN_SARIMA_POINTS: continue
    try: ps[va.index.get_indexer(idx)] = SARIMAX(y,order=SARIMA_ORDER,seasonal_order=SEASONAL_ORDER,enforce_stationarity=False,enforce_invertibility=False).fit(disp=False,maxiter=50).forecast(len(idx))
    except Exception: pass
scores={}
for a in np.arange(0,1.01,.05): scores[round(a,2)]=wmae(va.Weekly_Sales,a*px+(1-a)*ps,va.IsHoliday)
alpha=min(scores,key=scores.get); pred=alpha*px+(1-alpha)*ps
print({'xgboost':wmae(va.Weekly_Sales,px,va.IsHoliday),'sarima':wmae(va.Weekly_Sales,ps,va.IsHoliday),'hybrid':scores[alpha],'xgb_weight':alpha})
joblib.dump({'xgb':xgb,'xgb_weight':alpha,'sarima_order':SARIMA_ORDER,'seasonal_order':SEASONAL_ORDER,'min_points':MIN_SARIMA_POINTS},OUTPUT_DIR/'xgboost_sarima_baseline.joblib')